# Análise Exploratória: Mortes e Violências contra LGBTI+ no Brasil (2021–2024)

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import geopandas as gpd
import requests
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)


## 1. Carregamento e unificação das bases

In [ ]:
BASE_DIR = 'C:/Users/gabri/OneDrive/Desktop/observatorio/'

arquivos = {
    2021: BASE_DIR + 'Registro de Mortes Violentas de LGBTI+ no Brasil - 2021 - Registros.csv',
    2022: BASE_DIR + 'Sistematizaçao das Mortes e Violências contra LGBTI+ no Brasil - 2022 - Registros.csv',
    2023: BASE_DIR + 'Sistematização das Mortes e Violências contra LGBTI+ no Brasil - 2023 - Registros Final.csv',
    2024: BASE_DIR + 'Sistematizaçao das Mortes e Violências contra LGBTI+ no Brasil - 2024 - Antra 2024.csv',
}

RENAME = {
    'MACRORREGIÃO': 'REGIÃO',
    'MACROREGIÃO': 'REGIÃO',
    'COR/RAÇA': 'RAÇA/ETNIA',
    'Justificativa': 'JUSTIFICATIVA DO QUALIFICADOR',
}

dfs = []
for ano, caminho in arquivos.items():
    df = pd.read_csv(caminho, encoding='utf-8', low_memory=False, dtype=str)
    df = df.rename(columns=RENAME)
    df.insert(0, 'ANO', str(ano))
    dfs.append(df)

base = pd.concat(dfs, ignore_index=True, sort=False)
print(f'Base unificada: {base.shape[0]} registros | {base.shape[1]} colunas')


## 2. Limpeza e padronização

In [ ]:
# ─── Padroniza N.I. ───────────────────────────────────────────────────────────
def normaliza_ni(s):
    if pd.isna(s): return 'N.I.'
    s = str(s).strip()
    if s.upper() in ['N.I.', 'N.I', 'NI', 'N/I', '']: return 'N.I.'
    return s

colunas_categoricas = [
    'UF', 'LOCAL', 'ESPAÇO', 'PERÍODO (matutino, vespertino e noturno)',
    'ORIENTAÇÃO SEXUAL', 'IDENTIDADE DE GÊNERO', 'SEGMENTO',
    'RAÇA/ETNIA', 'PROFISSÃO/OCUPAÇÃO DA VÍTIMA', 'TIPIFICAÇÃO'
]

for col in colunas_categoricas:
    if col in base.columns:
        base[col] = base[col].apply(normaliza_ni).str.strip().str.title()

# ─── UF normalizada ───────────────────────────────────────────────────────────
uf_map = {
    'São Paulo': 'SP', 'Rio De Janeiro': 'RJ', 'Minas Gerais': 'MG',
    'Bahia': 'BA', 'Ceará': 'CE', 'Pernambuco': 'PE', 'Paraná': 'PR',
    'Rio Grande Do Sul': 'RS', 'Goiás': 'GO', 'Maranhão': 'MA',
    'Pará': 'PA', 'Amazonas': 'AM', 'Mato Grosso': 'MT',
    'Mato Grosso Do Sul': 'MS', 'Espírito Santo': 'ES',
    'Rio Grande Do Norte': 'RN', 'Paraíba': 'PB', 'Alagoas': 'AL',
    'Sergipe': 'SE', 'Piauí': 'PI', 'Tocantins': 'TO',
    'Rondônia': 'RO', 'Acre': 'AC', 'Amapá': 'AP', 'Roraima': 'RR',
    'Santa Catarina': 'SC', 'Distrito Federal': 'DF',
}
base['UF'] = base['UF'].replace(uf_map)

uf_map_upper = {
    'ACRE':'AC','ALAGOAS':'AL','AMAPÁ':'AP','AMAZONAS':'AM','BAHIA':'BA',
    'CEARÁ':'CE','DISTRITO FEDERAL':'DF','ESPÍRITO SANTO':'ES','GOIÁS':'GO',
    'MARANHÃO':'MA','MATO GROSSO':'MT','MATO GROSSO DO SUL':'MS',
    'MINAS GERAIS':'MG','PARÁ':'PA','PARAÍBA':'PB','PARANÁ':'PR',
    'PERNAMBUCO':'PE','PIAUÍ':'PI','RIO DE JANEIRO':'RJ',
    'RIO GRANDE DO NORTE':'RN','RIO GRANDE DO SUL':'RS','RONDÔNIA':'RO',
    'RORAIMA':'RR','SANTA CATARINA':'SC','SÃO PAULO':'SP','SERGIPE':'SE',
    'TOCANTINS':'TO',
}
base['UF_NORM'] = (
    base['UF'].str.strip().str.upper()
    .map(lambda x: uf_map_upper.get(x, x))
    .replace({'N.I': pd.NA, 'N.I.': pd.NA})
)

# ─── ESPAÇO ────────────────────────────────────────────────────────────────────
base['ESPAÇO'] = base['ESPAÇO'].replace({
    'Espaço Público': 'Público', 'Espaço Privado': 'Privado',
    'Espaco Público': 'Público', 'Espaco Privado': 'Privado',
    'Espaço público': 'Público',
})

# ─── TIPIFICAÇÃO ───────────────────────────────────────────────────────────────
base['TIPIFICAÇÃO'] = base['TIPIFICAÇÃO'].replace({
    'Homicidio': 'Homicídio',
    'Suicidio': 'Suicídio',
    'Latrocinio': 'Latrocínio',
    'Lesão Corporal Seguida De Morte': 'Lesão Corporal Seguida De Morte',
})

# ─── CVLI: Crimes Violentos Letais Intencionais ───────────────────────────────
# Agrupa homicídio + latrocínio + lesão corporal seguida de morte
CVLI_TIPOS = {'Homicídio', 'Latrocínio', 'Lesão Corporal Seguida De Morte'}

def classifica_cvli(tipo):
    if tipo in ('N.I.', ''):
        return 'N.I.'
    if tipo in CVLI_TIPOS:
        return 'CVLI'
    return tipo

base['TIPIFICAÇÃO_CVLI'] = base['TIPIFICAÇÃO'].apply(classifica_cvli)

# ─── Correção de grafia das profissões ────────────────────────────────────────
profissao_correcao = {
    # Profissional do sexo
    'Profissional Do Sexo'           : 'Profissional do Sexo',
    'Trabalhadora Do Sexo'           : 'Profissional do Sexo',
    'Trabalhador Do Sexo'            : 'Profissional do Sexo',
    'Trabalhador(A) Do Sexo'         : 'Profissional do Sexo',
    'Garota De Programa'             : 'Profissional do Sexo',
    'Prostituta'                     : 'Profissional do Sexo',
    'Profissional Sexual'            : 'Profissional do Sexo',
    # Autônomo
    'Autonomo'                       : 'Autônomo/a',
    'Autonoma'                       : 'Autônomo/a',
    'Autônomo'                       : 'Autônomo/a',
    'Autônoma'                       : 'Autônomo/a',
    # Desempregado
    'Desempregado'                   : 'Desempregado/a',
    'Desempregada'                   : 'Desempregado/a',
    # Serviços gerais
    'Servico Gerais'                 : 'Serviços Gerais',
    'Serviço Gerais'                 : 'Serviços Gerais',
    'Servicos Gerais'                : 'Serviços Gerais',
    'Auxiliar De Serviços Gerais'    : 'Aux. de Serviços Gerais',
    # Doméstica/o
    'Domestica'                      : 'Doméstico/a',
    'Domestico'                      : 'Doméstico/a',
    'Doméstica'                      : 'Doméstico/a',
    # Aposentado
    'Aposentado'                     : 'Aposentado/a',
    'Aposentada'                     : 'Aposentado/a',
    # Catador
    'Catador'                        : 'Catador/a de Recicláveis',
    'Catadora'                       : 'Catador/a de Recicláveis',
    'Catador De Materiais Reciclaveis': 'Catador/a de Recicláveis',
    'Catador De Material Reciclavel' : 'Catador/a de Recicláveis',
    # Pedreiro
    'Pedreiro'                       : 'Pedreiro/a',
    'Pedreira'                       : 'Pedreiro/a',
    # Cabeleireiro
    'Cabeleireiro'                   : 'Cabeleireiro/a',
    'Cabeleireira'                   : 'Cabeleireiro/a',
    # Garçom
    'Garcom'                         : 'Garçom/Garçonete',
    'Garçom'                         : 'Garçom/Garçonete',
    'Garconete'                      : 'Garçom/Garçonete',
    'Garçonete'                      : 'Garçom/Garçonete',
    # Agricultor
    'Agricultor'                     : 'Agricultor/a',
    'Agricultora'                    : 'Agricultor/a',
    'Trabalhador Rural'              : 'Agricultor/a',
    'Trabalhadora Rural'             : 'Agricultor/a',
    # Policial / militar
    'Policia Militar'                : 'Policial Militar',
    'Pm'                             : 'Policial Militar',
    # Diarista
    'Diarista'                       : 'Diarista',
    # Manicure
    'Manicure'                       : 'Manicure/Pedicure',
    'Pedicure'                       : 'Manicure/Pedicure',
    # Cozinheiro
    'Cozinheiro'                     : 'Cozinheiro/a',
    'Cozinheira'                     : 'Cozinheiro/a',
    # Comerciante
    'Comerciante'                    : 'Comerciante',
    # Moto-entregador
    'Motoboy'                        : 'Motoentregador/a',
    'Moto Boy'                       : 'Motoentregador/a',
    'Motoentregador'                 : 'Motoentregador/a',
}

base['PROFISSÃO/OCUPAÇÃO DA VÍTIMA'] = (
    base['PROFISSÃO/OCUPAÇÃO DA VÍTIMA'].replace(profissao_correcao)
)

# ─── Dados numéricos e temporais ──────────────────────────────────────────────
base['IDADE_NUM'] = pd.to_numeric(base['IDADE'], errors='coerce')
base['DATA MORTE'] = pd.to_datetime(base['DATA MORTE'], dayfirst=True, errors='coerce')
base['MÊS_NUM'] = base['DATA MORTE'].dt.month
base['FAIXA_ETARIA'] = pd.cut(
    base['IDADE_NUM'],
    bins=[0, 17, 24, 34, 44, 54, 100],
    labels=['< 18', '18–24', '25–34', '35–44', '45–54', '55+']
)

print('Limpeza concluída.')
print(f'  CVLI criados: {(base["TIPIFICAÇÃO_CVLI"] == "CVLI").sum()} registros')
base[colunas_categoricas].head(3)


## 3. Visão geral

In [ ]:
print('=== Registros por ano ===')
print(base['ANO'].value_counts().sort_index().to_frame('registros'))

print('\n=== Missing values nas colunas principais ===')
miss = base[colunas_categoricas + ['IDADE_NUM']].apply(
    lambda c: (c == 'N.I.' if c.dtype == object else c.isna()).sum()
)
miss_pct = (miss / len(base) * 100).round(1)
print(pd.DataFrame({'N.I. / NaN': miss, '%': miss_pct}))


In [ ]:
# Taxa de N.I. por variável — gráfico comparativo
variaveis_ni = {
    'Orientação Sexual'     : 'ORIENTAÇÃO SEXUAL',
    'Identidade de Gênero'  : 'IDENTIDADE DE GÊNERO',
    'Raça/Etnia'            : 'RAÇA/ETNIA',
    'Profissão/Ocupação'    : 'PROFISSÃO/OCUPAÇÃO DA VÍTIMA',
    'Local'                 : 'LOCAL',
    'Espaço'                : 'ESPAÇO',
    'Período do Dia'        : 'PERÍODO (matutino, vespertino e noturno)',
    'Segmento'              : 'SEGMENTO',
    'Tipificação'           : 'TIPIFICAÇÃO',
}

ni_pct = {
    label: (base[col] == 'N.I.').mean() * 100
    for label, col in variaveis_ni.items()
    if col in base.columns
}
ni_s = pd.Series(ni_pct).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(ni_s.index, ni_s.values, color='#e07b54', edgecolor='white', linewidth=1.1)
for bar, val in zip(bars, ni_s.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9, fontweight='bold')
ax.set_xlabel('% de registros com N.I. (não informado)')
ax.set_title('Taxa de não identificação por variável (2021–2024)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlim(0, ni_s.max() * 1.18)
sns.despine()
plt.tight_layout()
plt.show()


## 4. Evolução anual

In [ ]:
por_ano = base['ANO'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(por_ano.index, por_ano.values,
              color=sns.color_palette('muted', 4), edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, por_ano.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3, str(val),
            ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.set_title('Registros de mortes e violências por ano', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Ano')
ax.set_ylabel('Nº de registros')
ax.set_ylim(0, por_ano.max() * 1.15)
sns.despine()
plt.tight_layout()
plt.show()


## 5. Distribuição por UF (Top 15)

In [ ]:
top_uf = base['UF_NORM'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=top_uf.values, y=top_uf.index, palette='flare', ax=ax)
for i, v in enumerate(top_uf.values):
    ax.text(v + 1, i, str(v), va='center', fontsize=10)
ax.set_title('Top 15 estados com mais registros (2021–2024)', fontsize=13, fontweight='bold')
ax.set_xlabel('Nº de registros')
ax.set_ylabel('')
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
# Mapa de calor por estado
url = 'https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson'
gdf_raw = gpd.GeoDataFrame.from_features(requests.get(url).json()['features'])
gdf_raw['sigla'] = gdf_raw['sigla'].str.strip().str.upper()

contagem_uf = base['UF_NORM'].value_counts().reset_index()
contagem_uf.columns = ['sigla', 'registros']
gdf = gdf_raw.merge(contagem_uf, on='sigla', how='left')
gdf['registros'] = gdf['registros'].fillna(0)

fig, ax = plt.subplots(figsize=(12, 10))
gdf.plot(
    column='registros', cmap='YlOrRd', linewidth=0.5, edgecolor='white',
    legend=True,
    legend_kwds={'label': 'Nº de registros', 'orientation': 'horizontal', 'shrink': 0.5},
    ax=ax,
)
for _, row in gdf.iterrows():
    if row.geometry:
        x, y = row.geometry.centroid.x, row.geometry.centroid.y
        ax.annotate(
            f"{row['sigla']}\n{int(row['registros'])}",
            xy=(x, y), ha='center', va='center',
            fontsize=7, color='#222222', fontweight='bold'
        )
ax.set_title('Registros de mortes e violências LGBTI+ por estado (2021–2024)',
             fontsize=13, fontweight='bold', pad=15)
ax.axis('off')
plt.tight_layout()
plt.show()


## 6. Local e espaço da ocorrência

In [ ]:
import re

local_map = {
    r'(?i)^casa$'                    : 'Residência',
    r'(?i)casa\s*da\s*v[ií]tima'    : 'Residência',
    r'(?i)resid[eê]ncia.*'           : 'Residência',
    r'(?i)domic[ií]lio'              : 'Residência',
    r'(?i)^ap(artamento)?\.?$'      : 'Residência',
}

base['LOCAL_NORM'] = base['LOCAL'].copy()
for pattern, replacement in local_map.items():
    base['LOCAL_NORM'] = base['LOCAL_NORM'].str.replace(pattern, replacement, regex=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_local = (base[~base['LOCAL_NORM'].isin(['N.I.', 'N.I'])]
             ['LOCAL_NORM'].value_counts().head(10))
sns.barplot(x=top_local.values, y=top_local.index, palette='crest', ax=axes[0])
for i, v in enumerate(top_local.values):
    axes[0].text(v + 0.5, i, str(v), va='center', fontsize=9)
axes[0].set_title('Local da ocorrência (Top 10)', fontweight='bold')
axes[0].set_xlabel('Nº de registros')
axes[0].set_ylabel('')

espaco = base[base['ESPAÇO'].isin(['Público', 'Privado'])]['ESPAÇO'].value_counts()
axes[1].pie(espaco.values, labels=espaco.index, autopct='%1.1f%%',
            colors=['#e07b54', '#5b8db8'], startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Espaço da ocorrência', fontweight='bold')

sns.despine(ax=axes[0])
plt.tight_layout()
plt.show()


## 7. Período do dia

In [ ]:
col_periodo = 'PERÍODO (matutino, vespertino e noturno)'
periodo = base[base[col_periodo] != 'N.I.'][col_periodo].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(periodo.index, periodo.values,
              color=['#2c3e50', '#e67e22', '#f1c40f', '#95a5a6'][:len(periodo)],
              edgecolor='white')
for bar, val in zip(bars, periodo.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(val),
            ha='center', va='bottom', fontweight='bold')
ax.set_title('Período do dia das ocorrências', fontsize=13, fontweight='bold')
ax.set_ylabel('Nº de registros')
ax.set_ylim(0, periodo.max() * 1.15)
sns.despine()
plt.tight_layout()
plt.show()


## 8. Identidade de gênero, orientação sexual e segmento

In [ ]:
# ── Contagens incluindo N.I. ─────────────────────────────────────────────────
ig_all   = base['IDENTIDADE DE GÊNERO'].value_counts()
os_all   = base['ORIENTAÇÃO SEXUAL'].value_counts()
seg_all  = base['SEGMENTO'].value_counts()

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Identidade de gênero, orientação sexual e segmento (excl. N.I.)',
             fontsize=14, fontweight='bold', y=1.01)

# — Identidade de Gênero —
ig = ig_all[ig_all.index != 'N.I.']
sns.barplot(x=ig.values, y=ig.index, palette='rocket', ax=axes[0])
for i, v in enumerate(ig.values):
    axes[0].text(v + 0.5, i, str(v), va='center', fontsize=9)
axes[0].set_title('Identidade de Gênero', fontweight='bold')
axes[0].set_xlabel('Nº de registros')
axes[0].set_ylabel('')

# — Orientação Sexual —
os_sem_ni = os_all[os_all.index != 'N.I.']
sns.barplot(x=os_sem_ni.values, y=os_sem_ni.index, palette='magma', ax=axes[1])
for i, v in enumerate(os_sem_ni.values):
    axes[1].text(v + 0.5, i, str(v), va='center', fontsize=9)
axes[1].set_title('Orientação Sexual', fontweight='bold')
axes[1].set_xlabel('Nº de registros')
axes[1].set_ylabel('')

# — Segmento LGBTI+ —
seg = seg_all[seg_all.index != 'N.I.'].head(8)
sns.barplot(x=seg.values, y=seg.index, palette='mako', ax=axes[2])
for i, v in enumerate(seg.values):
    axes[2].text(v + 0.5, i, str(v), va='center', fontsize=9)
axes[2].set_title('Segmento LGBTI+', fontweight='bold')
axes[2].set_xlabel('Nº de registros')
axes[2].set_ylabel('')

sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
# ── Com N.I. visível — para avaliar subnotificação por categoria ──────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Distribuição incl. N.I. — identificação dos registros',
             fontsize=14, fontweight='bold', y=1.01)

for ax, (col, title) in zip(axes, [
    ('IDENTIDADE DE GÊNERO', 'Identidade de Gênero'),
    ('ORIENTAÇÃO SEXUAL',    'Orientação Sexual'),
    ('SEGMENTO',             'Segmento LGBTI+'),
]):
    counts = base[col].value_counts().head(10)
    cores  = ['#cccccc' if x == 'N.I.' else '#5b8db8' for x in counts.index]
    ax.barh(counts.index[::-1], counts.values[::-1], color=cores[::-1], edgecolor='white')
    for i, v in enumerate(counts.values[::-1]):
        ax.text(v + 0.3, i, str(v), va='center', fontsize=9)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Nº de registros')
    ax.set_ylabel('')

ni_patch  = mpatches.Patch(color='#cccccc', label='N.I. (não informado)')
val_patch = mpatches.Patch(color='#5b8db8', label='Identificado')
fig.legend(handles=[val_patch, ni_patch], loc='lower center', ncol=2,
           fontsize=10, bbox_to_anchor=(0.5, -0.04))

sns.despine()
plt.tight_layout()
plt.show()


## 8.5 Análise de N.I.: taxas por ano e correlação entre variáveis

In [ ]:
# ── Taxa de N.I. por variável e ano ──────────────────────────────────────────
variaveis_ni = {
    'Orient. Sexual'  : 'ORIENTAÇÃO SEXUAL',
    'Ident. Gênero'   : 'IDENTIDADE DE GÊNERO',
    'Raça/Etnia'      : 'RAÇA/ETNIA',
    'Profissão'       : 'PROFISSÃO/OCUPAÇÃO DA VÍTIMA',
    'Local'           : 'LOCAL',
    'Espaço'          : 'ESPAÇO',
    'Período'         : 'PERÍODO (matutino, vespertino e noturno)',
    'Segmento'        : 'SEGMENTO',
    'Tipificação'     : 'TIPIFICAÇÃO',
}

ni_ano = pd.DataFrame()
for label, col in variaveis_ni.items():
    if col in base.columns:
        ni_ano[label] = base.groupby('ANO')[col].apply(
            lambda s: (s == 'N.I.').mean() * 100
        )

fig, ax = plt.subplots(figsize=(13, 5))
ni_ano.T.plot(kind='bar', ax=ax, colormap='tab10', edgecolor='white', width=0.75)
ax.set_ylabel('% N.I.')
ax.set_title('Taxa de N.I. por variável e ano', fontsize=13, fontweight='bold')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=25)
ax.legend(title='Ano', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
# ── Correlação entre ausência de informação (N.I.) nas variáveis ──────────────
ni_bin = pd.DataFrame({
    label: (base[col] == 'N.I.').astype(int)
    for label, col in variaveis_ni.items()
    if col in base.columns
})
ni_bin['ANO (num)'] = base['ANO'].astype(int)

corr = ni_bin.corr()

# Máscara para diagonal
mask = np.eye(len(corr), dtype=bool)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
    linewidths=0.5, ax=ax, mask=mask,
    vmin=-1, vmax=1,
    annot_kws={'size': 8}
)
ax.set_title('Correlação entre ausência de informação (N.I.) nas variáveis',
             fontsize=13, fontweight='bold', pad=12)
plt.xticks(rotation=35, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

print('Pares com maior correlação positiva (subnotificação conjunta):')
corr_stack = corr.where(~mask).stack().sort_values(ascending=False)
print(corr_stack.head(10).round(3).to_string())


## 9. Raça/Etnia

In [ ]:
# ── Gráfico de barras com e sem N.I. ─────────────────────────────────────────
raca_all = base['RAÇA/ETNIA'].value_counts()
raca_sem = raca_all[raca_all.index != 'N.I.']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

cores_raca_barra = ['#f5e6c8', '#8B4513', '#d4a76a', '#228B22', '#4a4a4a', '#FFD700']

bars = axes[0].bar(raca_sem.index, raca_sem.values,
                   color=cores_raca_barra[:len(raca_sem)],
                   edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, raca_sem.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, str(val),
                 ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Raça/Etnia (excl. N.I.)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Nº de registros')
axes[0].set_ylim(0, raca_sem.max() * 1.15)

# Com N.I. visible
cores_ni = ['#cccccc' if x == 'N.I.' else '#C8874E' for x in raca_all.index]
axes[1].bar(raca_all.index, raca_all.values, color=cores_ni, edgecolor='white', linewidth=1.2)
for i, (idx, val) in enumerate(zip(raca_all.index, raca_all.values)):
    axes[1].text(i, val + 1, str(val), ha='center', va='bottom', fontweight='bold', fontsize=9)
axes[1].set_title('Raça/Etnia (incl. N.I.)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Nº de registros')
axes[1].set_ylim(0, raca_all.max() * 1.2)
axes[1].tick_params(axis='x', rotation=15)

ni_patch  = mpatches.Patch(color='#cccccc', label='N.I.')
val_patch = mpatches.Patch(color='#C8874E', label='Identificado')
axes[1].legend(handles=[val_patch, ni_patch], fontsize=9)

sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
# ── Mapa choropleth: raça/etnia predominante por estado ──────────────────────
# Calcula raca_predominante por UF (excluindo N.I.)
raca_uf = (
    base[~base['RAÇA/ETNIA'].isin(['N.I.'])]
    .groupby(['UF_NORM', 'RAÇA/ETNIA'])
    .size()
    .reset_index(name='n')
)
idx_max = raca_uf.groupby('UF_NORM')['n'].idxmax()
raca_predom = raca_uf.loc[idx_max, ['UF_NORM', 'RAÇA/ETNIA']].rename(
    columns={'UF_NORM': 'sigla', 'RAÇA/ETNIA': 'raca_predominante'}
)

# Re-usa gdf_raw (carregado na seção 5)
gdf_raca = gdf_raw.merge(contagem_uf, on='sigla', how='left')
gdf_raca = gdf_raca.merge(raca_predom, on='sigla', how='left')
gdf_raca['registros'] = gdf_raca['registros'].fillna(0)

# Paleta por raça — dicionário correto
CORES_RACA = {
    'Negra'    : '#5C3317',
    'Parda'    : '#C8874E',
    'Branca'   : '#F5E6C8',
    'Indígena' : '#3D8B3D',
    'Amarela'  : '#FFD700',
}

gdf_raca['cor'] = gdf_raca['raca_predominante'].map(CORES_RACA).fillna('#cccccc')

fig, ax = plt.subplots(figsize=(12, 10))
gdf_raca.plot(color=gdf_raca['cor'], linewidth=0.5, edgecolor='white', ax=ax)
ax.set_title('Raça/etnia predominante das vítimas LGBTI+ por estado (2021–2024)',
             fontweight='bold', fontsize=13)
ax.axis('off')

for _, row in gdf_raca.iterrows():
    if row.geometry:
        x, y = row.geometry.centroid.x, row.geometry.centroid.y
        label = row['raca_predominante'] if pd.notna(row['raca_predominante']) else '?'
        ax.annotate(f"{row['sigla']}\n{label}",
                    xy=(x, y), ha='center', va='center',
                    fontsize=6.5, color='#111111', fontweight='bold')

patches = [mpatches.Patch(color=c, label=r) for r, c in CORES_RACA.items()]
patches.append(mpatches.Patch(color='#cccccc', label='Sem dados'))
ax.legend(handles=patches, loc='lower left', fontsize=10,
          title='Raça predominante', title_fontsize=10, framealpha=0.9)
plt.tight_layout()
plt.show()


In [ ]:
# ── Alternativa 1: Heatmap UF × Raça/Etnia (Top 10 UFs) ─────────────────────
top10_uf = base['UF_NORM'].value_counts().head(10).index
heat_df = (
    base[base['UF_NORM'].isin(top10_uf) & (base['RAÇA/ETNIA'] != 'N.I.')]
    .groupby(['UF_NORM', 'RAÇA/ETNIA'])
    .size()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(heat_df, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.4, ax=ax, cbar_kws={'label': 'Nº de registros'})
ax.set_title('Raça/Etnia por UF — Top 10 estados (2021–2024)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Raça/Etnia')
ax.set_ylabel('UF')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# ── Alternativa 2: Evolução de Raça/Etnia ao longo dos anos ──────────────────
raca_ano = (
    base[base['RAÇA/ETNIA'] != 'N.I.']
    .groupby(['ANO', 'RAÇA/ETNIA'])
    .size()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(10, 5))
raca_ano.plot(kind='bar', ax=ax, colormap='tab10', edgecolor='white', width=0.75)
ax.set_title('Raça/Etnia das vítimas por ano', fontsize=13, fontweight='bold')
ax.set_xlabel('Ano')
ax.set_ylabel('Nº de registros')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Raça/Etnia', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
sns.despine()
plt.tight_layout()
plt.show()


## 10. Distribuição de idade

In [ ]:
idades = base['IDADE_NUM'].dropna()
idades = idades[(idades >= 10) & (idades <= 80)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(idades, bins=20, color='#5b8db8', edgecolor='white', linewidth=0.8)
axes[0].axvline(idades.mean(),   color='red',    linestyle='--', label=f'Média: {idades.mean():.1f}')
axes[0].axvline(idades.median(), color='orange', linestyle='--', label=f'Mediana: {idades.median():.1f}')
axes[0].set_title('Distribuição de idade das vítimas', fontweight='bold')
axes[0].set_xlabel('Idade')
axes[0].set_ylabel('Frequência')
axes[0].legend()

faixas = base['FAIXA_ETARIA'].value_counts().sort_index()
axes[1].bar(faixas.index.astype(str), faixas.values,
            color=sns.color_palette('flare', len(faixas)), edgecolor='white')
for i, v in enumerate(faixas.values):
    axes[1].text(i, v + 1, str(v), ha='center', va='bottom', fontweight='bold')
axes[1].set_title('Vítimas por faixa etária', fontweight='bold')
axes[1].set_xlabel('Faixa etária')
axes[1].set_ylabel('Nº de vítimas')

sns.despine()
plt.tight_layout()
plt.show()

print(f'Média de idade: {idades.mean():.1f} anos')
print(f'Mediana: {idades.median():.1f} anos')
print(f'Mínima: {int(idades.min())} | Máxima: {int(idades.max())}')


## 11. Tipificação e CVLI

In [ ]:
# ── Tipificação detalhada ─────────────────────────────────────────────────────
tip = base[base['TIPIFICAÇÃO'] != 'N.I.']['TIPIFICAÇÃO'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(x=tip.values, y=tip.index, palette='Spectral', ax=ax)
for i, v in enumerate(tip.values):
    ax.text(v + 1, i, str(v), va='center', fontsize=10)
ax.set_title('Tipificação das mortes e violências', fontsize=13, fontweight='bold')
ax.set_xlabel('Nº de registros')
ax.set_ylabel('')
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
# ── CVLI: Homicídio + Latrocínio + Lesão Corporal Seguida de Morte ────────────
cvli_contagem = base['TIPIFICAÇÃO_CVLI'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Barras: CVLI vs demais categorias
cores_cvli = ['#c0392b' if x == 'CVLI' else ('#cccccc' if x == 'N.I.' else '#5b8db8')
              for x in cvli_contagem.index]
axes[0].barh(cvli_contagem.index[::-1], cvli_contagem.values[::-1],
             color=cores_cvli[::-1], edgecolor='white')
for i, v in enumerate(cvli_contagem.values[::-1]):
    axes[0].text(v + 0.5, i, str(v), va='center', fontsize=9)
axes[0].set_title('Tipificação agrupada (CVLI vs demais)', fontweight='bold')
axes[0].set_xlabel('Nº de registros')
cvli_patch  = mpatches.Patch(color='#c0392b', label='CVLI')
out_patch   = mpatches.Patch(color='#5b8db8', label='Não-CVLI')
ni_patch    = mpatches.Patch(color='#cccccc', label='N.I.')
axes[0].legend(handles=[cvli_patch, out_patch, ni_patch], fontsize=9)

# Evolução do CVLI por ano
cvli_ano = (
    base[base['TIPIFICAÇÃO_CVLI'] == 'CVLI']
    .groupby('ANO')
    .size()
)
axes[1].plot(cvli_ano.index, cvli_ano.values, marker='o', color='#c0392b',
             linewidth=2.5, markersize=8)
for x, y in zip(cvli_ano.index, cvli_ano.values):
    axes[1].text(x, y + 1, str(y), ha='center', va='bottom', fontweight='bold', fontsize=11)
axes[1].set_title('Evolução dos CVLI por ano', fontweight='bold')
axes[1].set_xlabel('Ano')
axes[1].set_ylabel('Nº de registros CVLI')
axes[1].set_ylim(0, cvli_ano.max() * 1.2)

sns.despine()
plt.tight_layout()
plt.show()

# Composição do CVLI
cvli_comp = base[base['TIPIFICAÇÃO_CVLI'] == 'CVLI']['TIPIFICAÇÃO'].value_counts()
print('\n=== Composição do CVLI ===')
print(cvli_comp.to_frame('registros'))
print(f'Total CVLI: {cvli_comp.sum()} ({cvli_comp.sum()/len(base)*100:.1f}% do total)')


## 12. Profissão/Ocupação

> As grafias foram normalizadas na etapa de limpeza (seção 2). O mapeamento completo está no dicionário `profissao_correcao`.

In [ ]:
prof = (
    base[base['PROFISSÃO/OCUPAÇÃO DA VÍTIMA'] != 'N.I.']
    ['PROFISSÃO/OCUPAÇÃO DA VÍTIMA']
    .value_counts()
    .head(12)
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=prof.values, y=prof.index, palette='crest', ax=ax)
for i, v in enumerate(prof.values):
    ax.text(v + 0.3, i, str(v), va='center', fontsize=10)
ax.set_title('Profissão/Ocupação das vítimas (Top 12)', fontsize=13, fontweight='bold')
ax.set_xlabel('Nº de registros')
ax.set_ylabel('')
sns.despine()
plt.tight_layout()
plt.show()


## 13. Sazonalidade — ocorrências por mês

In [ ]:
meses_nomes = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
sazon = base.groupby(['ANO', 'MÊS_NUM']).size().reset_index(name='count')
sazon = sazon.dropna(subset=['MÊS_NUM'])
sazon['MÊS_NUM'] = sazon['MÊS_NUM'].astype(int)

fig, ax = plt.subplots(figsize=(12, 5))
for ano, grp in sazon.groupby('ANO'):
    ax.plot(grp['MÊS_NUM'], grp['count'], marker='o', label=str(ano), linewidth=2)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(meses_nomes)
ax.set_title('Ocorrências mensais por ano', fontsize=13, fontweight='bold')
ax.set_xlabel('Mês')
ax.set_ylabel('Nº de registros')
ax.legend(title='Ano')
sns.despine()
plt.tight_layout()
plt.show()


## 14. Cruzamentos

In [ ]:
# Identidade de gênero × Raça/Etnia
cross = pd.crosstab(
    base[base['IDENTIDADE DE GÊNERO'] != 'N.I.']['IDENTIDADE DE GÊNERO'],
    base[base['RAÇA/ETNIA'] != 'N.I.']['RAÇA/ETNIA']
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(cross, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Identidade de Gênero × Raça/Etnia', fontsize=13, fontweight='bold')
ax.set_xlabel('Raça/Etnia')
ax.set_ylabel('Identidade de Gênero')
plt.tight_layout()
plt.show()


In [ ]:
# Tipificação × Espaço
cross2 = pd.crosstab(
    base[base['TIPIFICAÇÃO'] != 'N.I.']['TIPIFICAÇÃO'],
    base[base['ESPAÇO'].isin(['Público', 'Privado'])]['ESPAÇO']
).head(8)

cross2.plot(kind='bar', figsize=(10, 4),
            color=['#5b8db8', '#e07b54'], edgecolor='white')
plt.title('Tipificação × Espaço da ocorrência', fontsize=13, fontweight='bold')
plt.xlabel('Tipificação')
plt.ylabel('Nº de registros')
plt.xticks(rotation=35, ha='right')
plt.legend(title='Espaço')
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
# Orientação Sexual × Identidade de Gênero (sem N.I.)
cross3 = pd.crosstab(
    base[base['ORIENTAÇÃO SEXUAL'] != 'N.I.']['ORIENTAÇÃO SEXUAL'],
    base[base['IDENTIDADE DE GÊNERO'] != 'N.I.']['IDENTIDADE DE GÊNERO']
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(cross3, annot=True, fmt='d', cmap='Blues', linewidths=0.4, ax=ax)
ax.set_title('Orientação Sexual × Identidade de Gênero', fontsize=13, fontweight='bold')
ax.set_xlabel('Identidade de Gênero')
ax.set_ylabel('Orientação Sexual')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()
